In [ ]:
from pathlib import Path
import json, os, re
from typing import Any, Dict, Optional
from openai import OpenAI

# 输入 / 输出路径
INPUT_JSON = ""
OUTPUT_JSON = INPUT_JSON.replace(".json", "_judgement.json")  # 与输入同文件夹

# JSON 字段名
BUGFREE_TRACE = "bug-free-trace"  # 读取 trace 的字段

# OpenAI
MODEL_NAME = "gpt-5"
api_key = ""  # 请替换为你自己的 API key
client = OpenAI(api_key=api_key)


In [7]:
# %% 重新定义（修复）模板 —— 只需运行这个 cell 覆盖原变量
# %% 重新定义 Step1 Prompt（召回优先 & 保持 is_bug + bugs）
SYSTEM_PROMPT = """You are a mobile-app QA oracle.

## 1.  Strict definition you MUST follow
NCF bug  ≜  A repeatable user action that, according to product rules or platform
conventions, should succeed but instead leaves the app in an **incorrect data/state**
WITHOUT crashing.  Visual polish, wording, confirmation dialogs, banner reminders,
or sluggish updates are usually *not* NCF bugs.

### Mark an NCF bug TRUE if at least one of these categories is triggered
F-1  State-transition failure        – list/record not inserted, deleted, or updated  
F-2  Persistence failure             – change vanishes after navigate/refresh  
F-3  Cross-screen inconsistency      – setting changed in screen A not reflected in B  
F-4  Wrong computation / value       – wrong unit, number, timestamp, grouping  
F-5  Mis-routed / unavailable control– tap does nothing or wrong thing  
F-6  Localization/I18N failure       – chosen locale not fully applied  
F-7  Accessibility regression        – action blocks screen-reader focus, etc.

### NOT NCF (design / UX) examples 
• **Informational banners, snack-bars, or toasts** that persist in the trace  
  - e.g. “Changes are not backed up”, “Outdated Android WebView”  
• **Telemetry or time-based values that drift on their own**  
  - data-usage counters, CPU %, clocks, GPS readouts, live tickers, etc.  
• **Background or automatic refreshes** while user does nothing  
  - list reorder when sync completes, badge counts changing, etc.  
• **Absence of extra confirmation** for standard actions (Paste, Delete, Sort, etc.)  
  unless the spec explicitly requires it.  
• **Place-holder or hint text** that is supposed to disappear after input  
  (“••••••”, “Enter note here…”, “No items yet”).  
• **Hidden-password dots** until the user taps “Show”.  
• **Duplicate view nodes in the dump** caused by RecyclerView / UIAutomator quirks  
  (identical `bounds` or `NAF` attributes).  
• **Menu or drawer labels that do not change** when a filter remains “All”.  
• **Values shown in equivalent formats** (00:00 vs 12:00 AM, 1 ft vs 0.3048 m).  
• **Speculative intent mismatches**  
  - the model must not assume the user *meant* “1:37 PM” if they tapped “AM”.  
• **Navigation to system or external screens** that looks unusual but is by design  
  (e.g. tapping “Salary” opens Android’s *InstalledAppDetails*).  
• **Settings pages that do not update until a restart / reopen**  
  (common for Theme, Language, Font-size on many Android apps).  
• **Search boxes left open** or **filters that yield “0 results”** — perfectly valid.  
• **Slow or incremental rendering** (cards loaded one-by-one, markdown preview after Save).  
• **Weak-password warnings** that still allow progress when the product permits it.  
• **Collapsible items mistaken for separate pages** (“Entities” expands in place).  
• **Lists whose final order coincidentally matches the previous order** even after Sort.  
• **Currency / locale availability** complaints (missing JPY, etc.) unless app claims support.  
• **“ERROR code 0 / success” messages** — success path, not a functional failure.  
• **Any claim that relies on hallucinated elements** not present in the provided trace.

(If the observed issue matches ONLY one of the bullets above and none of the F-categories,  
`is_ncf_bug` **must** be false.)

## 2.  What you must output

A JSON object **only**, no extra text:

```json
{
  "is_ncf_bug": <true | false>,
  "categories": [ "F-n", ... ],     // empty if false
  "evidence": [                     // up to 3 short strings
    "<quote the key screen fragments or actions that prove the bug>"
  ],
  "explanation": "<≤ 60 words why the criteria fire or why not>"
}
```

"""

EXAMPLE_PROMPT = """
UI info interaction trace example:

Example 1:
**UI info interaction trace:
*Initial Structure

*Current Screen Information:  #Current Activity: .DeckPicker.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Sync (log in)'}, {'ImageButton': 'Open drawer'}, 'AnkiDroid'];2#.[{'ImageButton': 'com.ichi2.anki:id/fab_main'}];Other Widgets with Text in This Page has the following group(s):1#.Collection is empty;2#.Start adding cards
using the + icon.;.
'action': 'click', 'feature': 'com.ichi2.anki:id/fab_main'

*Current Screen Information:  #Current Activity: .DeckPicker.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Sync (log in)'}, {'ImageButton': 'Open drawer'}, 'AnkiDroid'];2#.fabBGLayout:[{'View': 'com.ichi2.anki:id/fabBGLayout'}];3#.add_shared_layout:[{'ImageButton': 'com.ichi2.anki:id/add_shared_action'}, 'Get shared decks'];4#.add_filtered_deck_layout:[{'ImageButton': 'com.ichi2.anki:id/add_filtered_deck_action'}, 'Create filtered deck'];5#.add_deck_layout:[{'ImageButton': 'com.ichi2.anki:id/add_deck_action'}, 'Create deck'];6#.[{'ImageButton': 'com.ichi2.anki:id/fab_main'}, 'Add'];Other Widgets with Text in This Page has the following group(s):1#.Collection is empty;2#.Start adding cards
using the + icon.;.
'action': 'click', 'feature': 'Add'

*Current Screen Information:  #Current Activity: .NoteEditor.  # UI Information:set_text has the following group(s):1#.constraint_layout:[{'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Front field'}, {'ImageButton': 'Make field Front sticky'}, 'Front'];2#.constraint_layout:[{'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Back field'}, {'ImageButton': 'Make field Back sticky'}, 'Back'];click has the following group(s):1#.key_pos_header_voice:[{'FrameLayout': 'Voice input'}];2#.['Close features menu'];3#.['Search'];4#.['Sticker Keyboard'];5#.['GIF Keyboard'];6#.['Translate'];7#.['More features'];8#.key_pos_0_0:[{'FrameLayout': 'Q'}, '1'];9#.key_pos_0_1:[{'FrameLayout': 'W'}, '2'];10#.key_pos_0_2:[{'FrameLayout': 'E'}, '3'];11#.key_pos_0_3:[{'FrameLayout': 'R'}, '4'];12#.key_pos_0_4:[{'FrameLayout': 'T'}, '5'];13#.key_pos_0_5:[{'FrameLayout': 'Y'}, '6'];14#.key_pos_0_6:[{'FrameLayout': 'U'}, '7'];15#.key_pos_0_7:[{'FrameLayout': 'I'}, '8'];16#.key_pos_0_8:[{'FrameLayout': 'O'}, '9'];17#.key_pos_0_9:[{'FrameLayout': 'P'}, '0'];18#.['A'];19#.key_pos_1_1:[{'FrameLayout': 'S'}];20#.key_pos_1_2:[{'FrameLayout': 'D'}];21#.key_pos_1_3:[{'FrameLayout': 'F'}];22#.key_pos_1_4:[{'FrameLayout': 'G'}];23#.key_pos_1_5:[{'FrameLayout': 'H'}];24#.key_pos_1_6:[{'FrameLayout': 'J'}];25#.key_pos_1_7:[{'FrameLayout': 'K'}];26#.['L'];27#.key_pos_shift:[{'FrameLayout': 'Shift'}];28#.key_pos_2_1:[{'FrameLayout': 'Z'}];29#.key_pos_2_2:[{'FrameLayout': 'X'}];30#.key_pos_2_3:[{'FrameLayout': 'C'}];31#.key_pos_2_4:[{'FrameLayout': 'V'}];32#.key_pos_2_5:[{'FrameLayout': 'B'}];33#.key_pos_2_6:[{'FrameLayout': 'N'}];34#.key_pos_2_7:[{'FrameLayout': 'M'}];35#.key_pos_del:[{'FrameLayout': 'Delete'}];36#.key_pos_switch_to_symbol:[{'FrameLayout': 'Symbol keyboard'}];37#.key_pos_bottom_symbol_1:[{'FrameLayout': ','}];38#.key_pos_switch_to_next_language:[{'FrameLayout': 'Emoji button'}];39#.['Space'];40#.key_pos_bottom_symbol_2:[{'FrameLayout': '.'}];41#.['Enter'];42#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Preview'}, {'Button': 'Save'}, {'ImageButton': 'Navigate up'}, 'Add'];43#.CardEditorTagButton:['Tags: '];44#.CardEditorCardsButton:['Cards: Card 1'];45#.editor_toolbar:[{'ImageButton': 'Create Toolbar Item'}, {'ImageButton': 'Insert MathJax Equation'}, {'ImageButton': 'Change Font Size'}, {'ImageButton': 'Insert Heading'}, {'ImageButton': 'Insert Horizontal Line'}, {'ImageButton': 'Format as Underline'}, {'ImageButton': 'Format as Italic'}, {'ImageButton': 'Format as Bold'}];spinner has the following group(s):1#.note_deck_spinner:[{'Spinner': 'com.ichi2.anki:id/note_deck_spinner'}, 'Default'];scrollable has the following group(s):1#.note_type_spinner:[{'Spinner': 'com.ichi2.anki:id/note_type_spinner'}, 'Basic'];Other Widgets with Text in This Page has the following group(s):1#.Type:;2#.Deck:;.
'action': 'set_text', 'feature': 'Front', 'input_text': '<a href="https://google.com">test</a>'

*Current Screen Information:  #Current Activity: .NoteEditor.  # UI Information:set_text has the following group(s):1#.constraint_layout:['<a href="https://google.com">test</a>', {'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Front field'}, {'ImageButton': 'Make field Front sticky'}, 'Front'];2#.constraint_layout:[{'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Back field'}, {'ImageButton': 'Make field Back sticky'}, 'Back'];click has the following group(s):1#.['a'];2#.['to'];3#.['I'];4#.key_pos_header_voice:[{'FrameLayout': 'Voice input'}];5#.['Open features menu'];6#.key_pos_0_0:[{'FrameLayout': 'q'}, '1'];7#.key_pos_0_1:[{'FrameLayout': 'w'}, '2'];8#.key_pos_0_2:[{'FrameLayout': 'e'}, '3'];9#.key_pos_0_3:[{'FrameLayout': 'r'}, '4'];10#.key_pos_0_4:[{'FrameLayout': 't'}, '5'];11#.key_pos_0_5:[{'FrameLayout': 'y'}, '6'];12#.key_pos_0_6:[{'FrameLayout': 'u'}, '7'];13#.key_pos_0_7:[{'FrameLayout': 'i'}, '8'];14#.key_pos_0_8:[{'FrameLayout': 'o'}, '9'];15#.key_pos_0_9:[{'FrameLayout': 'p'}, '0'];16#.['a'];17#.key_pos_1_1:[{'FrameLayout': 's'}];18#.key_pos_1_2:[{'FrameLayout': 'd'}];19#.key_pos_1_3:[{'FrameLayout': 'f'}];20#.key_pos_1_4:[{'FrameLayout': 'g'}];21#.key_pos_1_5:[{'FrameLayout': 'h'}];22#.key_pos_1_6:[{'FrameLayout': 'j'}];23#.key_pos_1_7:[{'FrameLayout': 'k'}];24#.['l'];25#.key_pos_shift:[{'FrameLayout': 'Shift'}];26#.key_pos_2_1:[{'FrameLayout': 'z'}];27#.key_pos_2_2:[{'FrameLayout': 'x'}];28#.key_pos_2_3:[{'FrameLayout': 'c'}];29#.key_pos_2_4:[{'FrameLayout': 'v'}];30#.key_pos_2_5:[{'FrameLayout': 'b'}];31#.key_pos_2_6:[{'FrameLayout': 'n'}];32#.key_pos_2_7:[{'FrameLayout': 'm'}];33#.key_pos_del:[{'FrameLayout': 'Delete'}];34#.key_pos_switch_to_symbol:[{'FrameLayout': 'Symbol keyboard'}];35#.key_pos_bottom_symbol_1:[{'FrameLayout': ','}];36#.key_pos_switch_to_next_language:[{'FrameLayout': 'Emoji button'}];37#.['Space'];38#.key_pos_bottom_symbol_2:[{'FrameLayout': '.'}];39#.['Enter'];40#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Preview'}, {'Button': 'Save'}, {'ImageButton': 'Navigate up'}, 'Add'];41#.CardEditorTagButton:['Tags: '];42#.CardEditorCardsButton:['Cards: Card 1'];43#.editor_toolbar:[{'ImageButton': 'Create Toolbar Item'}, {'ImageButton': 'Insert MathJax Equation'}, {'ImageButton': 'Change Font Size'}, {'ImageButton': 'Insert Heading'}, {'ImageButton': 'Insert Horizontal Line'}, {'ImageButton': 'Format as Underline'}, {'ImageButton': 'Format as Italic'}, {'ImageButton': 'Format as Bold'}];spinner has the following group(s):1#.note_deck_spinner:[{'Spinner': 'com.ichi2.anki:id/note_deck_spinner'}, 'Default'];scrollable has the following group(s):1#.note_type_spinner:[{'Spinner': 'com.ichi2.anki:id/note_type_spinner'}, 'Basic'];Other Widgets with Text in This Page has the following group(s):1#.Type:;2#.Deck:;.
'action': 'click', 'feature': 'Save'

*Current Screen Information:  #Current Activity: .NoteEditor.  # UI Information:set_text has the following group(s):1#.constraint_layout:[{'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Front field'}, {'ImageButton': 'Make field Front sticky'}, 'Front'];2#.constraint_layout:[{'EditText': 'com.ichi2.anki:id/id_note_editText'}, {'ImageButton': 'com.ichi2.anki:id/id_expand_button'}, {'ImageButton': 'Attach multimedia content to the Back field'}, {'ImageButton': 'Make field Back sticky'}, 'Back'];click has the following group(s):1#.key_pos_header_voice:[{'FrameLayout': 'Voice input'}];2#.['Close features menu'];3#.['Search'];4#.['Sticker Keyboard'];5#.['GIF Keyboard'];6#.['Translate'];7#.['More features'];8#.key_pos_0_0:[{'FrameLayout': 'Q'}, '1'];9#.key_pos_0_1:[{'FrameLayout': 'W'}, '2'];10#.key_pos_0_2:[{'FrameLayout': 'E'}, '3'];11#.key_pos_0_3:[{'FrameLayout': 'R'}, '4'];12#.key_pos_0_4:[{'FrameLayout': 'T'}, '5'];13#.key_pos_0_5:[{'FrameLayout': 'Y'}, '6'];14#.key_pos_0_6:[{'FrameLayout': 'U'}, '7'];15#.key_pos_0_7:[{'FrameLayout': 'I'}, '8'];16#.key_pos_0_8:[{'FrameLayout': 'O'}, '9'];17#.key_pos_0_9:[{'FrameLayout': 'P'}, '0'];18#.['A'];19#.key_pos_1_1:[{'FrameLayout': 'S'}];20#.key_pos_1_2:[{'FrameLayout': 'D'}];21#.key_pos_1_3:[{'FrameLayout': 'F'}];22#.key_pos_1_4:[{'FrameLayout': 'G'}];23#.key_pos_1_5:[{'FrameLayout': 'H'}];24#.key_pos_1_6:[{'FrameLayout': 'J'}];25#.key_pos_1_7:[{'FrameLayout': 'K'}];26#.['L'];27#.key_pos_shift:[{'FrameLayout': 'Shift'}];28#.key_pos_2_1:[{'FrameLayout': 'Z'}];29#.key_pos_2_2:[{'FrameLayout': 'X'}];30#.key_pos_2_3:[{'FrameLayout': 'C'}];31#.key_pos_2_4:[{'FrameLayout': 'V'}];32#.key_pos_2_5:[{'FrameLayout': 'B'}];33#.key_pos_2_6:[{'FrameLayout': 'N'}];34#.key_pos_2_7:[{'FrameLayout': 'M'}];35#.key_pos_del:[{'FrameLayout': 'Delete'}];36#.key_pos_switch_to_symbol:[{'FrameLayout': 'Symbol keyboard'}];37#.key_pos_bottom_symbol_1:[{'FrameLayout': ','}];38#.key_pos_switch_to_next_language:[{'FrameLayout': 'Emoji button'}];39#.['Space'];40#.key_pos_bottom_symbol_2:[{'FrameLayout': '.'}];41#.['Enter'];42#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Preview'}, {'Button': 'Save'}, {'ImageButton': 'Navigate up'}, 'Add'];43#.CardEditorTagButton:['Tags: '];44#.CardEditorCardsButton:['Cards: Card 1'];45#.editor_toolbar:[{'ImageButton': 'Create Toolbar Item'}, {'ImageButton': 'Insert MathJax Equation'}, {'ImageButton': 'Change Font Size'}, {'ImageButton': 'Insert Heading'}, {'ImageButton': 'Insert Horizontal Line'}, {'ImageButton': 'Format as Underline'}, {'ImageButton': 'Format as Italic'}, {'ImageButton': 'Format as Bold'}];spinner has the following group(s):1#.note_deck_spinner:[{'Spinner': 'com.ichi2.anki:id/note_deck_spinner'}, 'Default'];scrollable has the following group(s):1#.note_type_spinner:[{'Spinner': 'com.ichi2.anki:id/note_type_spinner'}, 'Basic'];Other Widgets with Text in This Page has the following group(s):1#.Type:;2#.Deck:;.
'action': 'click', 'feature': 'Navigate up'

*Current Screen Information:  #Current Activity: .DeckPicker.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Sync (log in)'}, {'ImageButton': 'Open drawer'}, 'AnkiDroid', '1 card due'];2#.['com.ichi2.anki:id/DeckPickerHoriz'];3#.counts_layout:[{'LinearLayout': 'Open the deck overview page containing the number of cards to see today.'}, '1', '0', '0'];4#.[{'ImageButton': 'com.ichi2.anki:id/fab_main'}];Other Widgets with Text in This Page has the following group(s):1#.Default;2#.Studied ⁨0⁩ cards ⁨in ⁨0⁩ seconds⁩ today (⁨0⁩s/card);.
'action': 'click', 'feature': 'Default'

*Current Screen Information:  #Current Activity: .Reviewer.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Flag card'}, {'NAF': '[705,105][837,237]'}, {'ImageButton': 'Open drawer'}];2#.qa:['test'];3#.['com.ichi2.anki:id/touch_layer'];4#.flashcard_layout_flip:[{'FrameLayout': 'com.ichi2.anki:id/flashcard_layout_flip'}, 'Show answer'];Other Widgets with Text in This Page has the following group(s):1#.1;2#.0;3#.0;4#.AnkiDroid Flashcard;.
'action': 'click', 'feature': 'test'

*Current Screen Information:  #Current Activity: .Reviewer.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Flag card'}, {'NAF': '[705,105][837,237]'}, {'ImageButton': 'Open drawer'}];2#.qa:['test'];3#.['com.ichi2.anki:id/touch_layer'];4#.flashcard_layout_flip:[{'FrameLayout': 'com.ichi2.anki:id/flashcard_layout_flip'}, 'Show answer'];Other Widgets with Text in This Page has the following group(s):1#.1;2#.0;3#.0;4#.AnkiDroid Flashcard;.
'action': 'click', 'feature': 'test'

*Current Screen Information:  #Current Activity: .Reviewer.  # UI Information:click has the following group(s):1#.toolbar:[{'ImageView': 'More options'}, {'Button': 'Flag card'}, {'NAF': '[705,105][837,237]'}, {'ImageButton': 'Open drawer'}];2#.qa:['test'];3#.['com.ichi2.anki:id/touch_layer'];4#.flashcard_layout_flip:[{'FrameLayout': 'com.ichi2.anki:id/flashcard_layout_flip'}, 'Show answer'];Other Widgets with Text in This Page has the following group(s):1#.1;2#.0;3#.0;4#.AnkiDroid Flashcard;.

**output:

```json
{
  "is_ncf_bug": true,
  "categories": ["F-5"],     
  "evidence": [
    "'action': 'set_text', 'feature': 'Front', 'input_text': '<a href=""https://google.com"">test</a>'",
    "'action': 'click', 'feature': 'test'",
  ],
  "explanation": "The sequence shows that after creating a card with a hyperlink in the 'Front' field and saving it, the card is shown in the reviewer with the text 'test' instead of displaying the hyperlink as expected. This indicates that the hyperlink is not being rendered or displayed correctly in the card review, which is a functionality issue."
}
```
Done Example 1.


Example 2:
**UI info interaction trace:

*Initial Structure\n\n*Current Screen Information:  #Current Activity: design.codeux.authpass.MainActivity.  # UI Information:click has the following group(s):1#.[{'Button': 'Show menu'}, {'Button': 'Search'}, {'Button': 'Filter by group'}, {'Button': 'Save'}, {'Button': 'Open navigation menu'}, 'AuthPass'];2#.[{'Button': 'Create your first Password'}];Other Widgets with Text in This Page has the following group(s):1#...️;2#.You do not have any password in your database yet.;.\n'action': 'click', 'feature': 'Create your first Password'\n\n*Current Screen Information:  #Current Activity: design.codeux.authpass.MainActivity.  # UI Information:click has the following group(s):1#.[{'Button': 'Show menu'}, {'Button': 'Back'}, '(no title)'];2#.[{'View': '\\uf084'}];3#.[{'View': 'Group:\\nPersonalPasswords'}];4#.[{'Button': 'Add Field'}];5#.[{'Button': 'Add Attachment'}];scrollable has the following group(s):1#.[{'Button': 'Show menu'}, 'Title'];2#.[{'Button': 'Show menu'}, 'Website'];3#.[{'Button': 'Show menu'}, 'User'];4#.[{'Button': 'Show menu'}, {'Button': 'Generate Password (cmd+g)'}, 'Password'];Other Widgets with Text in This Page has the following group(s):1#.File:\nPersonalPasswords;2#.Last Modified:\n4/17/2025 17:18:19;.\n'action': 'set_text', 'feature': 'Title', 'input_text': 'testTitle'\n'action': 'set_text', 'feature': 'Website', 'input_text': 'uc.edu'\n'action': 'set_text', 'feature': 'User', 'input_text': 'testUser'\n'action': 'set_text', 'feature': 'Password', 'input_text': '123456'\n\n*Current Screen Information:  #Current Activity: design.codeux.authpass.MainActivity.  # UI Information:click has the following group(s):1#.[{'Button': 'Show menu'}, {'NAF': '[828,63][954,210]'}, {'Button': 'Back'}, '(no title)'];2#.[{'View': '\\uf084'}];3#.[{'View': 'Group:\\nPersonalPasswords'}];4#.[{'Button': 'Add Field'}];5#.[{'Button': 'Add Attachment'}];scrollable has the following group(s):1#.[{'Button': 'Show menu'}, 'testTitle, Title'];2#.[{'Button': 'Show menu'}, {'Button': 'Open URL (shift+cmd+U)'}, 'uc.edu, Website'];3#.[{'Button': 'Show menu'}, 'testUser, User'];4#.[{'Button': 'Show menu'}, '123456, Password'];Other Widgets with Text in This Page has the following group(s):1#.File:\nPersonalPasswords;2#.Last Modified:\n4/17/2025 17:19:51;.\n'action': 'click', 'feature': 'Save' at (891,137)\n\n*Current Screen Information:  #Current Activity: design.codeux.authpass.MainActivity.  # UI Information:click has the following group(s):1#.[{'Button': 'Show menu'}, {'Button': 'Back'}, 'testTitle'];2#.[{'View': '\\uf084'}];3#.[{'View': 'Group:\\nPersonalPasswords'}];4#.[{'Button': 'Add Field'}];5#.[{'Button': 'Add Attachment'}];scrollable has the following group(s):1#.[{'Button': 'Show menu'}, 'testTitle, Title'];2#.[{'Button': 'Show menu'}, {'Button': 'Open URL (shift+cmd+U)'}, 'uc.edu, Website'];3#.[{'Button': 'Show menu'}, 'testUser, User'];4#.[{'Button': 'Show menu'}, '123456, Password'];Other Widgets with Text in This Page has the following group(s):1#.File:\nPersonalPasswords;2#.Last Modified:\n4/17/2025 17:20:19;.\n'action': 'click', 'feature': 'back'\n\n*Current Screen Information:  #Current Activity: design.codeux.authpass.MainActivity.  # UI Information:click has the following group(s):1#.[{'Button': 'Show menu'}, {'Button': 'Search'}, {'Button': 'Filter by group'}, {'Button': 'Open navigation menu'}, 'AuthPass'];2#.[{'ImageView': 'testTitle\\ntestUser'}];3#.[{'Button': 'Add New Password'}];.\n'action': 'click', 'feature': 'testTitle'\n\n*Current Screen Information:  #Current Activity: design.codeux.authpass.MainActivity.  # UI Information:click has the following group(s):1#.[{'Button': 'Show menu'}, {'Button': 'Back'}, 'testTitle'];2#.[{'View': 'Group:\\nPersonalPasswords'}];3#.[{'Button': 'Add Field'}];4#.[{'Button': 'Add Attachment'}];scrollable has the following group(s):1#.[{'Button': 'Show menu'}, 'testTitle, Title'];2#.[{'Button': 'Show menu'}, {'Button': 'Open URL (shift+cmd+U)'}, 'uc.edu, Website'];3#.[{'Button': 'Show menu'}, 'testUser, User'];4#.[{'Button': 'Show menu'}, {'Button': 'Show protected field'}, {'Button': 'Protected field. Click to reveal.'}, 'Password\\n**********'];Other Widgets with Text in This Page has the following group(s):1#.File:\nPersonalPasswords;2#.Last Modified:\n4/17/2025 17:20:19;.\n

**output:

```json
{
  "is_ncf_bug": false,
  "categories": [],     
  "evidence": [],
  "explanation": "This is a standard security design pattern, not a data loss or persistence failure. The presence of a 'Show protected field' button confirms this is intentional behavior, which is explicitly listed as not an NCF bug."
}
```
Done Example 2.

• If several failures appear, list every triggering category.
• If no NCF bug, `is_ncf_bug` = false and leave `categories` empty.
• Keep evidence snippets short—just enough to make the verdict clear.

BEGIN EVALUATION NOW.

UI info interaction trace:

"""

In [8]:
def safe_json_loads(s: str) -> Optional[dict]:
    try:
        return json.loads(s)
    except Exception:
        pass
    fence = re.compile(r"^\s*```(?:json)?\s*([\s\S]*?)\s*```\s*$", re.IGNORECASE)
    m = fence.match(s or "")
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            return None
    return None


# %% 重新定义 Step1 调用（提示词说明召回优先）
def build_input_from_example_and_trace(trace_text: str) -> str:
    # 只加两句英语，强调输出与长度限制；不改动你的 SYSTEM_PROMPT 内容
    extra_note = (
        "Respond with a single JSON object exactly matching the schema. "
        "No code fences or extra text. Evidence ≤ 3 items; explanation ≤ 60 words.\n"
    )
    # EXAMPLE_PROMPT 末尾已含 “UI info interaction trace:” 提示，这里直接拼接 trace
    return (EXAMPLE_PROMPT or "").rstrip() +  (trace_text or "") + "\n" + extra_note

# 兜底：强制满足 schema
def enforce_schema(d: Any) -> Dict[str, Any]:
    out = {
        "is_ncf_bug": False,
        "categories": [],
        "evidence": [],
        "explanation": "",
    }
    if not isinstance(d, dict):
        return out

    # is_ncf_bug -> bool
    v = d.get("is_ncf_bug", False)
    if isinstance(v, bool):
        out["is_ncf_bug"] = v
    elif isinstance(v, str):
        out["is_ncf_bug"] = v.strip().lower() in {"true", "yes", "y", "1"}
    else:
        out["is_ncf_bug"] = bool(v)

    # categories -> list[str]，仅允许 F-<数字>
    cats = d.get("categories", [])
    if isinstance(cats, list):
        cats = [str(x).strip() for x in cats if str(x).strip()]
        cats = [c for c in cats if re.match(r"^F-\d+$", c)]
    else:
        cats = []
    out["categories"] = cats if out["is_ncf_bug"] else []

    # evidence -> list[str] (≤3)
    ev = d.get("evidence", [])
    if isinstance(ev, list):
        ev = [str(x).strip() for x in ev if str(x).strip()]
    else:
        ev = []
    out["evidence"] = ev[:3]

    # explanation -> ≤60 words
    exp = str(d.get("explanation", "") or "").strip()
    if exp:
        words = re.split(r"\s+", exp)
        if len(words) > 60:
            exp = " ".join(words[:60])
    out["explanation"] = exp

    return out

# %% 提取 reasoning.summary（兼容 list / str / 对象.text）
def extract_reasoning_summaries(resp) -> list[str]:
    summaries = []
    for out in getattr(resp, "output", []) or []:
        if getattr(out, "type", None) == "reasoning":
            summ = getattr(out, "summary", None)
            if isinstance(summ, list):
                for s in summ:
                    txt = getattr(s, "text", None) or (s.get("text") if isinstance(s, dict) else None)
                    if txt:
                        summaries.append(txt)
            else:
                txt = None
                if isinstance(summ, str):
                    txt = summ
                else:
                    txt = getattr(summ, "text", None)
                if txt:
                    summaries.append(txt)
    return summaries

def judge_trace_bugfree_with_reasoning(trace_text: str):
    input_payload = build_input_from_example_and_trace(trace_text)
    resp = client.responses.create(
        model=MODEL_NAME,
        instructions=SYSTEM_PROMPT,           # 你的 SYSTEM_PROMPT 作为系统指令
        input=input_payload,                  # 示例 + 轻量英文补充 + trace
        text={"format": {"type": "json_object"}},
        reasoning={"summary": "auto"},        # 要求返回 reasoning summary
    )

    reasoning_summaries = extract_reasoning_summaries(resp)
    raw_text = getattr(resp, "output_text", "") or ""
    data = safe_json_loads(raw_text) or {}
    parsed = enforce_schema(data)
    return parsed, raw_text, reasoning_summaries


In [9]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    items = json.load(f)

if not isinstance(items, list):
    raise ValueError("输入 JSON 顶层必须是 list。")

for i, item in enumerate(items, start=1):
    trace = item.get(BUGFREE_TRACE, "") or ""
    print(f"Proceeding {i}/{len(items)} trace_len={len(trace)}")

    if not trace.strip():
        parsed = {
            "is_ncf_bug": False,
            "categories": [],
            "evidence": [],
            "explanation": "Empty trace; cannot find an NCF bug.",
        }
        reasoning_summaries = []
    else:
        parsed, raw_text, reasoning_summaries = judge_trace_bugfree_with_reasoning(trace)

    # 结果 JSON：判定写入 judgement；同时保留推理摘要
    item["bugfree_judgement"] = parsed
    item["bugfree_reasoning"] = reasoning_summaries   # ← 新增字段：列表，每条为一段 summary 文本


    
    # if i >= 10:
    #     print("只处理前 10 条，提前结束。")
    #     break

# 保存（保持原结构，前 10 条已新增 judgement）
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(items, f, ensure_ascii=False, indent=2)

print(f"完成：已写入 {OUTPUT_JSON}")


Proceeding 1/600 trace_len=2740
Proceeding 2/600 trace_len=1927
Proceeding 3/600 trace_len=3473
Proceeding 4/600 trace_len=3989
Proceeding 5/600 trace_len=1480
Proceeding 6/600 trace_len=2875
Proceeding 7/600 trace_len=6145
Proceeding 8/600 trace_len=9548
Proceeding 9/600 trace_len=3531
Proceeding 10/600 trace_len=3288
Proceeding 11/600 trace_len=3236
Proceeding 12/600 trace_len=3414
Proceeding 13/600 trace_len=17376
Proceeding 14/600 trace_len=3954
Proceeding 15/600 trace_len=2277
Proceeding 16/600 trace_len=8483
Proceeding 17/600 trace_len=16532
Proceeding 18/600 trace_len=1704
Proceeding 19/600 trace_len=5037
Proceeding 20/600 trace_len=3550
Proceeding 21/600 trace_len=6720
Proceeding 22/600 trace_len=3987
Proceeding 23/600 trace_len=9870
Proceeding 24/600 trace_len=5178
Proceeding 25/600 trace_len=2737
Proceeding 26/600 trace_len=6773
Proceeding 27/600 trace_len=7022
Proceeding 28/600 trace_len=4458
Proceeding 29/600 trace_len=5694
Proceeding 30/600 trace_len=6235
Proceeding 31/600